# AI-Based Robotic Fruit Sorting System
## Canonical CNN Training Notebook
This notebook follows the 15-section architecture mandated by the Master Engineering Prompt.

### SECTION 1 — Environment Setup

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import random

# Random seed fixing for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# GPU check
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Directory creation
os.makedirs('./dataset', exist_ok=True)
os.makedirs('./exported_models', exist_ok=True)


### SECTION 2 — Dataset Loading

In [ ]:
# TODO: Insert logic to download from Mendeley / ScienceDirect Link
# Link: https://data.mendeley.com/datasets/bdd69gyhv8/1
# E.g., !wget <dataset_url>
# !unzip dataset.zip -d ./dataset


### SECTION 3 — Dataset Filtering

In [ ]:
import shutil

# Keep only required classes:
ALLOWED_CLASSES = [
    'fresh_apple', 'rotten_apple', 
    'fresh_grape', 'rotten_grape', 
    'fresh_strawberry', 'rotten_strawberry'
]

# Logic to remove unwanted directories (banana, guava, orange, pomegranate, jujube)
# dataset_dir = './dataset'
# for class_name in os.listdir(dataset_dir):
#     if class_name not in ALLOWED_CLASSES:
#         shutil.rmtree(os.path.join(dataset_dir, class_name))


### SECTION 4 — Dataset Validation

In [ ]:
# Image count verification & corrupt image removal
# Expected: ~200 original images per class
from pathlib import Path
from PIL import Image

def validate_dataset(dataset_dir):
    for class_name in ALLOWED_CLASSES:
        class_dir = Path(dataset_dir) / class_name
        if not class_dir.exists(): continue
        
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        valid_count = 0
        for img_path in images:
            try:
                img = Image.open(img_path)
                img.verify() # Verify image is not corrupt
                valid_count += 1
            except Exception:
                os.remove(img_path)
                print(f"Removed corrupt image: {img_path}")
        print(f"{class_name}: {valid_count} images")

# validate_dataset('./dataset')


### SECTION 5 — Train/Validation/Test Split
Target: 60% Train, 20% Validation, 20% Test

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
DATASET_DIR = './dataset'

# Split 1: 80% train+val, 20% test (Assuming splitting via tf.keras.utils.image_dataset_from_directory or custom folder split)
# Since image_dataset_from_directory supports validation_split, we can do 80/20 train/test, and split train again.
# A cleaner approach is to use split-folders library or tf.data.


### SECTION 6 — Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal_and_vertical"),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
  tf.keras.layers.RandomContrast(0.2)
])


### SECTION 7 — TensorFlow Data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare_pipeline(ds, augment=False):
    # Resizing and Normalization built into model or pipeline
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

# train_ds = prepare_pipeline(train_ds, augment=True)
# val_ds = prepare_pipeline(val_ds)
# test_ds = prepare_pipeline(test_ds)


### SECTION 8 — CNN Architecture

In [ ]:
model = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    
    Dense(6, activation='softmax')
])

model.summary()


### SECTION 9 — Training

In [ ]:
model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', # use sparse if labels are ints, categorical if one-hot
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint('./exported_models/best_model.h5', save_best_only=True)

# history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[early_stopping, checkpoint])


### SECTION 10 — Visualization

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.legend()
    plt.title('Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Loss')
    plt.show()

# plot_history(history)


### SECTION 11 — Evaluation

In [ ]:
# loss, accuracy = model.evaluate(test_ds)
# print(f"Test Accuracy: {accuracy*100:.2f}%")

# TODO: Confusion matrix and classification report using sklearn.metrics


### SECTION 12 — Unknown Fruit Detection

In [ ]:
def predict_fruit(img_array, model, class_names, threshold=0.7):
    predictions = model.predict(img_array)
    max_confidence = np.max(predictions)
    
    if max_confidence < threshold:
        return "Unknown Fruit", max_confidence
    else:
        predicted_class = class_names[np.argmax(predictions)]
        return predicted_class, max_confidence


### SECTION 13 — Inference Testing

In [ ]:
# Inference code to load a single image, resize, and call predict_fruit()


### SECTION 14 — Model Export

In [ ]:
# Export .h5 and SavedModel
model.save('./exported_models/final_model.h5')
model.save('./exported_models/final_model_saved')

# TFLite Export
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('./exported_models/model.tflite', 'wb') as f:
    f.write(tflite_model)

# Save Class Names
import json
with open('./exported_models/class_names.json', 'w') as f:
    json.dump(ALLOWED_CLASSES, f)


### SECTION 15 — ESP32 Integration Hooks

In [ ]:
import requests

ESP32_IP = "192.168.1.100"

def trigger_sorting_arm(predicted_class):
    # Example Action Mapping
    if 'fresh' in predicted_class:
        # Move to ACCEPT bin
        angles = {'base': 120, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    elif 'rotten' in predicted_class:
        # Move to REJECT bin
        angles = {'base': 60, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    else:
        # Do nothing or alert
        return
        
    url = f"http://{ESP32_IP}/move?base={angles['base']}&shoulder={angles['shoulder']}&elbow={angles['elbow']}&claw={angles['claw']}"
    try:
        print(f"Sending command to ESP32: {url}")
        # response = requests.get(url, timeout=2)
        # print("ESP32 Response:", response.status_code)
    except Exception as e:
        print("Failed to connect to ESP32:", e)
